# Оглавление

1. [Описание задачи](#описание-задачи)
2. [Описание исходных данных](#описание-исходных-данных)
3. [Фиксация виртуального окружения](#фиксация-виртуального-окружения)
4. [1. Предварительная обработка данных](#1-предварительная-обработка-данных)
   - [1.1. Чтение и загрузка данных](#11-чтение-и-загрузка-данных)
   - [1.2. Разделение выборки на обучение и тест](#12-разделение-выборки-на-обучение-и-тест)
   - [1.3. Первичный анализ данных](#13-первичный-анализ-данных)

# Описание задачи

## Предметная область

После падения кометы Созина и победы Аватара Аанга над Лордом огня Озаем гармония в мире начала восстанавливаться. На одной из удалённых территорий Царства Земли была обнаружена дикая стая летающих бизонов — ранее считалось, что большинство бизонов исчезло, остался только Аппа. Это открытие стало сенсацией.

Животные держатся подальше от людей, постоянно перемещаются и не помогают исследователям. Прямой способ изучить возраст бизонов невозможен — прибор для этого случайно сжег Зуко.

## Боли заказчика

Команда Аватара собрала множество наблюдений за каждым бизоном: особенности поведения, физические характеристики, показатели активности. Однако определить точный возраст бизона напрямую практически сложно — животные улетают при виде человека.

По словам Аанга, вес бизона — один из важных признаков, по которому можно судить о его возрасте. Монахи учили, что если хочешь понять животное, нужно научиться замечать, как оно движется и взаимодействует с окружающим миром.

## Задача ML

**Задача:** предсказать возраст летающего бизона по его физическим характеристикам и показателям активности.

**Объект:** летающий бизон (одна запись в датасете).

**Образ выходного результата модели:** число — возраст бизона (возраст в годах, непрерывная величина).

**Тип задачи:** регрессия.

## Подходы к решению

Согласно подсказке из МЕМО, можно рассмотреть два подхода:
1. **Прямая регрессия:** предсказывать Age напрямую по всем признакам.
2. **Двухэтапная схема:** сначала предсказать Whole_weight по признакам, затем Age по Whole_weight и другим признакам.

В рамках ЛР 1 мы рассмотрим оба подхода и сравним их на этапе обучения моделей (ЛР 5).

**Источник:** `materials/variants/05_avatar.md`

# Описание исходных данных

## Семантика и ожидаемые типы признаков

| Признак | Семантика | Ожидаемый тип | Целевой |
|---|---|---|---|
| Type | Тип бизона (исторические данные / дикие / прирученные) | categorical (string) | Нет |
| Length | Длина бизона | float | Нет |
| Diameter | Диаметр бизона | float | Нет |
| Height | Высота бизона | float | Нет |
| Whole_weight | Общий вес бизона | float | Нет |
| Age | Возраст бизона (в годах) | float | **Да** |

**Источник:** `materials/INDEX.md`, стр. 141; `materials/variants/05_avatar.md`

## Формулы предметной области

Формулы предметной области для предсказания возраста в датасете не предоставлены. МЕМО даёт только подсказку о возможной двухэтапной схеме: сначала предсказать вес по признакам, затем возраст по весу. Конкретные формулы связи веса и возраста не указаны — они будут определяться моделью ML на этапе обучения.

**Источник:** `materials/variants/05_avatar.md`, стр. 28-29

# Фиксация виртуального окружения

## Версия Python

Виртуальное окружение: `%USERPROFILE%\.venvs\studies2026-progpractice`

## Основные импорты

```python
import pandas as pd
import numpy as np
import typing as t
import warnings
warnings.filterwarnings('ignore')
```

## Случайный сид

```python
seed = 42
np.random.seed(seed)
```

**Источник:** `materials/INDEX.md`, стр. 97-99; `materials/lectures/Лекция 1 - check_lifelines.ipynb`, ячейки 1-2

In [ ]:
import pandas as pd
import numpy as np
import typing as t
import warnings
warnings.filterwarnings('ignore')

seed = 42
np.random.seed(seed)

# 1. Предварительная обработка данных

## 1.1. Чтение и загрузка данных

Загружаем датасет с характеристиками летающих бизонов. Проверяем корректность загрузки: количество строк и столбцов, типы данных, наличие пропусков.

**Источник:** `task.md`, стр. 43; `materials/INDEX.md`, стр. 93

In [ ]:
import os

# Путь к датасету (абсолютный путь через домашнюю директорию пользователя)
home_dir = os.path.expanduser('~')
data_path = os.path.join(home_dir, 'OneDrive', 'Desktop', 'Studies 2026', 'ProgPractice', 'materials', 'datasets', '05_avatar.csv')

print(f"Путь к датасету: {data_path}")
print(f"Файл существует: {os.path.exists(data_path)}")

# Загрузка данных (разделитель ',', UTF-8)
df = pd.read_csv(data_path, sep=',')

# Проверка корректности загрузки
print(f"Размер датасета: {df.shape}")
print(f"\nПервые 5 строк:")
df.head()

### Удаление строк с пропусками в таргете

Согласно INDEX.md, в признаке Age есть 5% пропусков. Поскольку Age — целевой признак, строки с пропусками в нём нужно удалить. Удаляем до разделения на train/test, чтобы избежать смещения распределений.

**Источник:** `materials/INDEX.md`, стр. 156

In [ ]:
# Удаление строк с пропусками в таргете Age (до разделения)
df = df.dropna(subset=['Age'])

print(f"Размер после удаления пропусков в Age: {df.shape}")

## 1.2. Разделение выборки на обучение и тест

Согласно лекции 1, сначала разделяем выборку на train и test, затем проводим анализ только на train. Это необходимо, чтобы избежать утечки данных из test в процесс обучения.

Для регрессии не используем `stratify=` (непрерывный таргет). Выбираем `test_size=0.2` (20% на тест).

**Источник:** `materials/INDEX.md`, стр. 94; `materials/lectures/Лекция 1 - check_lifelines.ipynb`, ячейка 8

In [ ]:
from sklearn.model_selection import train_test_split

# Разделение на train и test (20% на тест)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=seed)

print(f"Train размер: {train_df.shape}")
print(f"Test размер: {test_df.shape}")

## 1.3. Первичный анализ данных

Проводим анализ только на train-выборке, чтобы избежать утечки данных. Проверяем:
- Типы данных (соответствуют ожидаемым?)
- Количество уникальных значений (решение об удалении категорий)
- Наличие пропусков в каждом признаке
- Основные статистики (min, max, mean, median)

**Источник:** `materials/INDEX.md`, стр. 93; `materials/lectures/Лекция 1 - check_lifelines.ipynb`, ячейки 4-6

In [ ]:
# Информация о типах данных и пропусках
train_df.info()

In [ ]:
# Количество уникальных значений в каждом признаке
train_df.nunique()

In [ ]:
# Основные статистики
train_df.describe()

In [ ]:
# Уникальные значения категориального признака Type
print("Уникальные значения Type:")
print(train_df['Type'].value_counts())

### Анализ на утечки данных

Проверяем, нет ли признаков, которые неизвестны на момент прогноза (например, будущие даты, идентификаторы, признаки, вычисляемые из таргета). В датасете Avatar таких признаков нет — все физические характеристики измеряются одновременно с возрастом.

**Решение:** признаки для удаления: нет.

**Источник:** `materials/INDEX.md`, стр. 93; `materials/lectures/Лекция 1 - check_lifelines.ipynb`, ячейка 7